# Customer Segmentation & Marketing Campaign Analytics
## Phase 6 — Business Interpretation

**Goal of this notebook.** Turn Phases 3-4's numbers into a recommendation a marketing team could
actually act on: which segments to prioritize, which channel and timing to use, and where the
data's own limits mean a recommendation should be held loosely rather than followed blindly.

**A rule for this phase:** every recommendation below is checked against a possible alternative
explanation before it's stated as a finding. Two of the checks below (Sections 2 and 3) change the
practical advice from what a first look at the data would suggest.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.db import connect

pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
con = connect()
TARGET_SQL = "CASE WHEN subscribed THEN 1.0 ELSE 0 END"

## 1. Which segments to prioritize

Phase 4 already ranked the four segments by response rate and by how efficiently each one uses
calling effort. Restated as a priority order for a marketing team:

In [2]:
priority = con.execute('''
    SELECT s.segment_name, COUNT(*) AS n_customers,
           ROUND(100.0*AVG(''' + TARGET_SQL + '''),2) AS subscribe_rate_pct
    FROM bank_clean c JOIN customer_segments s USING (customer_id)
    GROUP BY 1 ORDER BY subscribe_rate_pct DESC
''').fetchdf()
priority

,segment_name,n_customers,subscribe_rate_pct
0,Warm: Past Success,1511,64.73
1,Warm: Past Mixed Outcome,1840,16.68
2,Warm: Past No,4903,12.60
3,Cold: Never Contacted,36957,9.16


**Recommendation, in order:**

1. **Warm: Past Success — call first, every time capacity allows.** 64.7% response, and Phase 4's
   efficiency index of 8.47 means this segment already returns far more than its share of calling
   effort. There is no plausible reason to under-call this group.
2. **Warm: Past Mixed Outcome and Warm: Past No — a solid second tier.** Both convert above the
   11.7% overall average (16.7% and 12.6%) and both have an efficiency index above 1. Call these
   after Past Success, not instead of it.
3. **Cold: Never Contacted — the segment to rethink, not to stop calling.** At 81.7% of all
   customers, this segment can't simply be dropped without abandoning most of the addressable
   market. But it converts below average (9.2%) and absorbed 86.4% of all calling effort for an
   efficiency index of 0.74. Sections 2 and 3 below narrow down *which* Cold contacts are worth
   the call.

## 2. Timing: does "March, September, October, December are best" hold up, or is it a fluke of
   which segments happen to get called then?

Phase 2 found these four months converting at 44-52%, against 6.7% in May. Because 82% of
customers are in the low-converting Cold segment, it's worth checking whether that timing effect
is real, or just an artifact of *which segment* happens to get called in which month.

In [3]:
timing_by_segment = con.execute('''
    SELECT s.segment_name,
           CASE WHEN c.month IN ('mar','sep','oct','dec') THEN 'Best 4 months'
                WHEN c.month = 'may' THEN 'May'
                ELSE 'Other months' END AS month_group,
           COUNT(*) AS n, ROUND(100.0*AVG(''' + TARGET_SQL.replace('subscribed','c.subscribed') + '''),2) AS rate
    FROM bank_clean c JOIN customer_segments s USING (customer_id)
    GROUP BY 1,2 ORDER BY 1, rate DESC
''').fetchdf()
timing_by_segment

,segment_name,month_group,n,rate
0,Cold: Never Contacted,Best 4 months,1063,42.71
1,Cold: Never Contacted,Other months,24642,9.38
2,Cold: Never Contacted,May,11252,5.51
3,Warm: Past Mixed Outcome,Best 4 months,186,40.86
4,Warm: Past Mixed Outcome,Other months,1063,16.65
5,Warm: Past Mixed Outcome,May,591,9.14
6,Warm: Past No,Best 4 months,385,34.55
7,Warm: Past No,Other months,2831,12.50
8,Warm: Past No,May,1687,7.77
9,Warm: Past Success,Best 4 months,374,74.06


**What this shows.** The Best-4-months effect holds inside **every single segment**, not just in
the aggregate: Cold jumps from 9.4% to 42.7%, Past No from 12.5% to 34.6%, Past Mixed from 16.7% to
40.9%, and even the already-strong Past Success segment rises from 64.5% to 74.1%. This is a real
timing effect, not a segment-mix illusion — worth building into a calling calendar regardless of
which segment is being called.

## 3. A second check: is "avoid May" actually about the month, or about how May was recorded?

Section 2 confirms months matter. But Phase 1 found that `contact = 'unknown'` is heavily
concentrated in certain months, and `contact = 'unknown'` converts far worse than `cellular`
(Phase 2). If "unknown" contacts are concentrated in May, part of May's poor showing could be a
recording-quality problem, not a real seasonal effect.

In [4]:
unknown_by_month = con.execute('''
    SELECT month,
           COUNT(*) FILTER (WHERE contact = 'unknown') AS n_unknown,
           COUNT(*) AS n_total,
           ROUND(100.0*COUNT(*) FILTER (WHERE contact = 'unknown')/COUNT(*),1) AS pct_unknown
    FROM bank_clean GROUP BY 1 ORDER BY pct_unknown DESC
''').fetchdf()
unknown_by_month

,month,n_unknown,n_total,pct_unknown
0,jun,4532,5341,84.9
1,may,7975,13766,57.9
2,sep,47,579,8.1
3,oct,50,738,6.8
4,jul,276,6895,4.0
5,mar,7,477,1.5
6,dec,3,214,1.4
7,nov,51,3970,1.3
8,aug,51,6247,0.8
9,jan,9,1403,0.6


**What this shows.** May and June are extreme outliers: **58% of May's contacts and 85% of June's
are recorded as `unknown` contact type**, versus under 9% for every other month. Every other month
in the dataset has largely reliable contact-type records; May and June do not. That is exactly
where the earlier "May is the worst month" finding needs a second look.

In [5]:
cellular_only_by_month = con.execute('''
    SELECT
        CASE WHEN month IN ('mar','sep','oct','dec') THEN 'Best 4 months'
             WHEN month IN ('may','jun') THEN 'May / June'
             ELSE 'Other months' END AS month_group,
        COUNT(*) AS n, ROUND(100.0*AVG(''' + TARGET_SQL + '''),2) AS rate
    FROM bank_clean WHERE contact = 'cellular'
    GROUP BY 1 ORDER BY rate DESC
''').fetchdf()
cellular_only_by_month

,month_group,n,rate
0,Best 4 months,1614,49.44
1,May / June,6060,15.73
2,Other months,21611,12.11


In [6]:
june_detail = con.execute('''
    SELECT month, COUNT(*) AS n, ROUND(100.0*AVG(''' + TARGET_SQL + '''),2) AS rate
    FROM bank_clean WHERE contact = 'cellular' AND month IN ('may','jun')
    GROUP BY 1
''').fetchdf()
june_detail

,month,n,rate
0,may,5331,11.87
1,jun,729,43.90


**What this shows, and why it changes the recommendation.** Restricting to customers reached on a
verified **cellular** number: the Best-4-months effect is still there and still strong (49.4% vs.
12.1% for other months) — that part of Section 2's finding survives the check. But **May and June
look completely different once "unknown"-channel noise is removed**: cellular contacts made in May
convert at **11.9%**, essentially the overall average, and cellular contacts made in **June convert
at 43.9%** — nearly as well as the officially "best" months. June's poor blended reputation
(10.2% in Phase 2) came almost entirely from the 90% of its contacts that were on an unreliable
line, not from June itself being a bad time to call.

**Revised timing recommendation:** treat March, September, October and December as genuinely
strong months, confirmed even after accounting for contact quality. Do **not** treat May, and
especially June, as bad months in general — they were bad mostly for customers reached on an
unreliable number. On a verified cellular number, June performs about as well as the best months.

## 4. Channel: does "cellular beats unknown" hold inside every segment?

In [7]:
channel_by_segment = con.execute('''
    SELECT s.segment_name, c.contact, COUNT(*) AS n,
           ROUND(100.0*AVG(''' + TARGET_SQL.replace('subscribed','c.subscribed') + '''),2) AS rate
    FROM bank_clean c JOIN customer_segments s USING (customer_id)
    GROUP BY 1,2 ORDER BY 1, n DESC
''').fetchdf()
channel_by_segment

,segment_name,contact,n,rate
0,Cold: Never Contacted,cellular,21732,12.04
1,Cold: Never Contacted,unknown,12950,4.00
2,Cold: Never Contacted,telephone,2275,11.08
3,Warm: Past Mixed Outcome,cellular,1637,17.10
4,Warm: Past Mixed Outcome,telephone,176,12.50
5,Warm: Past Mixed Outcome,unknown,27,18.52
6,Warm: Past No,cellular,4531,12.65
7,Warm: Past No,telephone,341,12.02
8,Warm: Past No,unknown,31,12.90
9,Warm: Past Success,cellular,1385,64.98


**What this shows.** For **Cold: Never Contacted**, where the sample sizes are large (12,950 to
21,732 per channel), the pattern is unambiguous: cellular (12.0%) and telephone (11.1%) both far
outperform unknown (4.0%). For the three warm segments the direction is less certain — Past No is
flat across channels, and Past Mixed's "unknown" cell (18.5%) is actually higher than cellular, but
on only 27 customers, far too few to trust. **The channel finding is solid for the Cold segment,
which is also where it matters most** (it is 82% of the customer base); for the warm segments,
there isn't enough data on the non-cellular channels to say anything with confidence.

## 5. Putting it together: a calling playbook

Combining Sections 1-4 into a simple set of rules, ordered by how much evidence supports each one.

In [8]:
playbook = pd.DataFrame([
    {"Rule": "Call Warm: Past Success first, whenever there is capacity", "Confidence": "High", "Evidence": "64.7% response, efficiency index 8.47, holds in every month and channel cut"},
    {"Rule": "Call Warm: Past Mixed Outcome and Past No next", "Confidence": "High", "Evidence": "Both above the 11.7% average and efficiency index above 1"},
    {"Rule": "Concentrate calling in March, September, October, December", "Confidence": "High", "Evidence": "Holds within every segment and within cellular-only contacts"},
    {"Rule": "For Cold: Never Contacted, insist on a cellular number before calling", "Confidence": "High (for this segment)", "Evidence": "12.0% cellular vs. 4.0% unknown, on 20,000+ customers each"},
    {"Rule": "Don't write off May or June — verify the number first", "Confidence": "Medium", "Evidence": "Cellular-only June converts at 43.9%; the raw month-level rate was masking this"},
    {"Rule": "Channel choice for the three warm segments", "Confidence": "Low — not enough data", "Evidence": "Non-cellular sample sizes of 12-176 customers per segment"},
])
playbook

,Rule,Confidence,Evidence
0,"Call Warm: Past Success first, whenever there ...",High,"64.7% response, efficiency index 8.47, holds i..."
1,Call Warm: Past Mixed Outcome and Past No next,High,Both above the 11.7% average and efficiency in...
2,"Concentrate calling in March, September, Octob...",High,Holds within every segment and within cellular...
3,"For Cold: Never Contacted, insist on a cellula...",High (for this segment),"12.0% cellular vs. 4.0% unknown, on 20,000+ cu..."
4,Don't write off May or June — verify the numbe...,Medium,Cellular-only June converts at 43.9%; the raw ...
5,Channel choice for the three warm segments,Low — not enough data,Non-cellular sample sizes of 12-176 customers ...


## 6. Honest limitations

- **One bank, one dataset.** This is a single Portuguese bank's telemarketing campaign for one
  product (a term deposit). Nothing here should be assumed to transfer to a different bank,
  product, or country without testing it there.
- **The data spans more than one year, but the file carries no year column.** UCI's own
  documentation for this dataset says it is "ordered by date," covering a multi-year period (the
  closely related `bank-additional-full.csv` version is documented as running from May 2008 to
  November 2010, which includes the 2008 financial crisis). Checking the row order in `bank-full.csv`
  itself confirms this: contacts labeled "may," for instance, appear scattered across the *entire*
  file rather than in one contiguous block, which is only possible if "May" recurs across multiple
  separate years. **A month label here is therefore an average across however many years the
  campaign ran, potentially mixing very different economic conditions under one name** — this
  campaign calendar can't be treated as "what May looks like" in any single year.
- **`duration` was deliberately excluded**, per the leakage warning in `data/README.md`. It is
  known to be one of the strongest predictors of subscription, so every recommendation above is
  working with less predictive information than would technically be available after a call has
  already started — appropriately, since the goal is deciding who to call *before* that happens.
- **This is association, not a controlled experiment.** No customer here was randomly assigned to
  be called or not, or to be called in one month versus another. "March converts better than
  May" describes what happened in this data, not a guaranteed result of moving calls to March —
  a bank running this playbook would want to test it before betting the whole campaign on it.
- **No cost or profit data.** There is no loan amount, deposit size, interest margin, or
  call-center cost in this dataset, so none of the recommendations above can be converted into a
  currency figure. "Worth calling" here means "converts at a good rate for the effort," not "is
  profitable" in any audited sense.
- **Small samples in several of the cuts above.** The channel breakdown for the three warm
  segments, and some individual month/segment combinations, involve a few dozen customers. Those
  specific percentages should be read as directional, not precise.

## Phase 6 summary

1. **Call Past Success customers first, always** — the highest response rate (64.7%) and the
   segment that already returns more than its share of effort.
2. **March, September, October and December are genuinely strong months** — confirmed even after
   controlling for which segment and which contact channel was used.
3. **May and June's poor reputation was partly a data-quality artifact.** Once restricted to
   verified cellular numbers, May looks average and June looks nearly as good as the best months.
   The real lesson from these two months is "verify the number," not "don't call."
4. **For the Cold: Never Contacted segment — 82% of all customers — insist on a cellular contact
   before calling.** The gap there (12.0% vs. 4.0%) is large and backed by tens of thousands of
   customers per channel.
5. **None of this is a controlled experiment**, the data may span several different years under
   one set of month labels, and `duration` was excluded on purpose — so this is a playbook to test
   deliberately, not a guarantee.